In [16]:
import os
from dotenv import load_dotenv
from dataclasses import dataclass
from openai import OpenAI

load_dotenv()

@dataclass(frozen=True)
class Provider():
    """ One provider described as pure DATA (same design as Notebook 1)."""
    name:str
    env_var:str
    is_free:bool
    base_url:str
    model:str

PROVIDERS = [
    Provider("OpenAI",     "OPENAI_API_KEY",     False, None,                              "gpt-4o-mini"),
    Provider("Groq",       "GROQ_API_KEY",       True,  "https://api.groq.com/openai/v1", "llama-3.3-70b-versatile"),
]

def select_provider()->Provider:
    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider
    expected = "/n".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No Provider key set. Add one of {expected} keys to .env file.")

def build_client(provider:Provider):
    provider = select_provider()
    api_key = os.environ.get(provider.env_var)
    if provider.base_url == None:
        return OpenAI(api_key=api_key)
    return OpenAI(base_url=provider.base_url, api_key=api_key)

def have_any_key()->bool:
    return any(os.environ.get(p.env_var for p in PROVIDERS))

def llm_reply(prompt:str, *, max_tokens:int=400)->str:
    """Send one user prompt; return the assistant's text."""
    provider = select_provider()
    client = build_client(provider)

    result = client.chat.completions.create(
        model=provider.model,   
        max_tokens=max_tokens,
        messages=[
            {"role":"user","content":prompt}
        ]
    )
    print(result.choices[0].message.content)



##### Zero Shot COT

In [17]:
ZERO_SHOT_COT_SUFFIX=f"\n\nLets think step by step"

def zero_shot_cot_prompt(question:str)->str:
    """ This function takes a question and returns the prompt for Zero shot COT model"""
    return f"Question: {question} {ZERO_SHOT_COT_SUFFIX}"

In [18]:
QUESTION = ("Roger has 5 tennis balls. He buys 2 more cans of tennis balls. "
    "Each can has 3 tennis balls. How many tennis balls does he have now?"
)
print(type(QUESTION))

<class 'str'>


In [19]:
print(zero_shot_cot_prompt(QUESTION))

Question: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now? 

Lets think step by step


In [20]:
reply = llm_reply(zero_shot_cot_prompt(QUESTION))
print(reply)

Let's break this down step by step:

1. **Initial Amount of Tennis Balls**: Roger starts with 5 tennis balls.

2. **Tennis Balls from Cans**: Roger buys 2 cans of tennis balls. Each can contains 3 tennis balls. 
   - To find out how many tennis balls are in the cans, we multiply the number of cans by the number of tennis balls per can:
     \[
     2 \text{ cans} \times 3 \text{ tennis balls/can} = 6 \text{ tennis balls}
     \]

3. **Total Number of Tennis Balls**: Now, we add the tennis balls he initially had to the tennis balls he bought:
   \[
   5 \text{ tennis balls} + 6 \text{ tennis balls} = 11 \text{ tennis balls}
   \]

Therefore, Roger now has **11 tennis balls**.
None


### Few Shot COT

In [21]:
FEW_SHOT_EXAMPLES = """
Q: There are 15 trees in the grove. Grove workers will plant 6 trees today. How many trees will be in the grove?
A: There are 15 trees to start. After planting 6 more, there are 15 + 6 = 21 trees.
Final answer: 21

Q: Leah had 32 chocolates and her sister had 42. They ate 35. How many pieces do they have left in total?
A: Leah and her sister had 32 + 42 = 74 chocolates together. After eating 35, they have 74 - 35 = 39 left.
Final answer: 39
""".strip()

In [22]:
def few_shot_cot_prompt(question:str)->str:
    """This function takes a question and returns a prompt for the few-shot COT model."""
    return f""" Solve each question by reasoning step by step.
        {FEW_SHOT_EXAMPLES}
        Question: {question}
        Answer: 
    """

In [23]:
reply = llm_reply(few_shot_cot_prompt(QUESTION))

print(reply)

To solve this problem, let's break it down step by step:

1. **Start with the initial number of tennis balls:** Roger has 5 tennis balls.

2. **Find out how many tennis balls are in each can:** Each can contains 3 tennis balls.

3. **Calculate the total number of tennis balls in the cans he buys:** Since Roger buys 2 cans, we multiply the number of cans by the number of balls per can:
   - 2 cans × 3 tennis balls per can = 6 tennis balls.

4. **Add the tennis balls from the cans to the initial amount:** Now we add the number of tennis balls Roger originally had to the number he just bought:
   - 5 tennis balls + 6 tennis balls = 11 tennis balls.

Final answer: 11
None


In [24]:
TRICKY = (
    "A juggler can juggle 16 balls. Half of the balls are golf balls, and "
    "half of the golf balls are blue. How many blue golf balls are there?"
)


In [28]:
reply = llm_reply(TRICKY)
print(reply)

Let's break down the problem step by step:

1. The total number of balls is 16.
2. Half of the balls are golf balls, so the number of golf balls is:

\[
\frac{16}{2} = 8
\]

3. Half of the golf balls are blue, so the number of blue golf balls is:

\[
\frac{8}{2} = 4
\]

Thus, the number of blue golf balls is \( \boxed{4} \).
None


In [26]:
reply = llm_reply(zero_shot_cot_prompt(TRICKY))
print(reply)

Let's break it down step by step:

1. **Total Number of Balls**: The juggler can juggle 16 balls in total.

2. **Golf Balls**: Half of these balls are golf balls. Since there are 16 balls, we calculate half of that:
   \[
   \frac{16}{2} = 8
   \]
   So, there are 8 golf balls.

3. **Blue Golf Balls**: Half of the golf balls are blue. Since there are 8 golf balls, we calculate half of that:
   \[
   \frac{8}{2} = 4
   \]
   Therefore, there are 4 blue golf balls.

Thus, the number of blue golf balls is **4**.
None


In [27]:
reply = llm_reply(few_shot_cot_prompt(TRICKY))
print(reply)

Let's break down the problem step by step.

1. The total number of balls the juggler can juggle is 16.
2. It is stated that half of the balls are golf balls. Therefore, the number of golf balls is:
   \[
   \frac{16}{2} = 8 \text{ golf balls.}
   \]
3. Next, we are told that half of the golf balls are blue. Therefore, the number of blue golf balls is:
   \[
   \frac{8}{2} = 4 \text{ blue golf balls.}
   \]

Final answer: 4
None
